In [10]:
import pandas as pd
import numpy as np
import networkx as nx
from dowhy import CausalModel

df = pd.read_csv("/home/snt/projects_lujun/LLMJudgeTempCausal/input/llm_judge_causal_data.csv")

# Step 1: One-Hot 编码，并统一列名格式
df_enc = pd.get_dummies(df, columns=["model_name", "judge_type", "prompt_strategy"])
df_enc.columns = [c.replace(" ", "_").replace("-", "_") for c in df_enc.columns]

# Step 2: 获取编码后的混淆变量列名
confounder_cols = [c for c in df_enc.columns 
                   if c.startswith(("model_name_", "judge_type_", "prompt_strategy_"))]
print("Confounders:", confounder_cols)

# Step 3: 重建 DAG（使用编码后列名）
edges = []
for c in confounder_cols:
    edges.append((c, "temperature"))
    edges.append((c, "error_rate"))
edges.append(("temperature", "error_rate"))

graph = nx.DiGraph(edges)

# Step 4: 构建因果模型
causal_model = CausalModel(
    data=df_enc,
    treatment="temperature",
    outcome="error_rate",
    graph=graph
)

# Step 5: 识别 + 估计
identified_estimand = causal_model.identify_effect(proceed_when_unidentifiable=True)

estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True
    # ⚠️ 不传 control_value/treatment_value，让 DoWhy 自动用均值计算 ATE
)
print(f"\n✅ 温度对 error_rate 的 ATE: {estimate.value:.4f}")
print(estimate)


Confounders: ['model_name_Claude_3_Haiku', 'model_name_GPT_3.5', 'model_name_GPT_4o', 'model_name_Llama_3_70B', 'model_name_Mistral_7B', 'judge_type_pairwise', 'judge_type_pointwise', 'prompt_strategy_chain_of_thought', 'prompt_strategy_direct', 'prompt_strategy_role_play']

✅ 温度对 error_rate 的 ATE: 0.1075
*** Causal Estimate ***

## Identified estimand
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
      d                                                                        ↪
──────────────(E[error_rate|model_name_GPT_4o,prompt_strategy_role_play,judge_ ↪
d[temperature]                                                                 ↪

↪                                                                              ↪
↪ type_pointwise,model_name_Claude_3_Haiku,judge_type_pairwise,model_name_Llam ↪
↪                                                                              ↪

↪                                               

In [11]:
# 反驳测试
refute_placebo = causal_model.refute_estimate(
    identified_estimand, estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute"
)
print("\n安慰剂测试（效应应接近 0）:")
print(refute_placebo)

refute_random = causal_model.refute_estimate(
    identified_estimand, estimate,
    method_name="random_common_cause"
)
print("\n随机混淆测试（效应应基本不变）:")
print(refute_random)



安慰剂测试（效应应接近 0）:
Refute: Use a Placebo Treatment
Estimated effect:0.10748780098834498
New effect:-0.0016068677218601902
p value:0.68


随机混淆测试（效应应基本不变）:
Refute: Add a random common cause
Estimated effect:0.10748780098834498
New effect:0.10746429093295784
p value:0.94

